# Demo 07 : Subagents

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · dict-subagents

A dict subagent is isolated. It receives the task description, not the parent messages. create_deep_agent also adds a general-purpose subagent when none is supplied under that name.

In [2]:
import json
import sqlite3
import threading
import time
from pathlib import Path

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_core.callbacks import BaseCallbackHandler
from deepagents import create_deep_agent
from deepagents.backends import StateBackend, FilesystemBackend, StoreBackend, CompositeBackend
from deepagents.backends.utils import create_file_data
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.reset()

    def reset(self):
        self.calls = self.input = self.output = self.invalid = self.errors = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for generations in response.generations:
                for generation in generations:
                    message = getattr(generation, "message", None)
                    usage = getattr(message, "usage_metadata", None) or {}
                    self.input += usage.get("input_tokens", 0)
                    self.output += usage.get("output_tokens", 0)
                    self.invalid += len(getattr(message, "invalid_tool_calls", []) or [])

    def on_llm_error(self, error, **kwargs):
        with self.lock:
            self.errors += 1


counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=1800, max_retries=0)
metrics = []
LIMIT = {"recursion_limit": 24}
DB = Path(data("demo-world/service.db")).resolve()
MANUALS = Path(data_folder("demo-world/manuals"))
PAGES = {p.name: p.read_text(encoding="utf-8") for p in sorted(MANUALS.glob("*.md"))}
WORKSPACE = (Path.cwd() / "workspace").resolve()
WORKSPACE.mkdir(exist_ok=True)


def ask(text):
    return {"messages": [HumanMessage(text)]}


def answer(result):
    return str(next(m.content for m in reversed(result["messages"])
                    if isinstance(m, AIMessage) and m.content and not m.tool_calls))


def measured(label, operation):
    counter.reset()
    start = time.perf_counter()
    try:
        return operation()
    finally:
        row = dict(label=label, calls=counter.calls, input_tokens=counter.input,
                   output_tokens=counter.output, malformed=counter.invalid, errors=counter.errors,
                   seconds=round(time.perf_counter() - start, 1))
        metrics.append(row)
        print("metrics :", json.dumps(row))


async def measured_async(label, operation):
    counter.reset()
    start = time.perf_counter()
    try:
        return await operation()
    finally:
        row = dict(label=label, calls=counter.calls, input_tokens=counter.input,
                   output_tokens=counter.output, malformed=counter.invalid, errors=counter.errors,
                   seconds=round(time.perf_counter() - start, 1))
        metrics.append(row)
        print("metrics :", json.dumps(row))


def show(result):
    print("tools :", [m.name for m in result["messages"] if m.type == "tool"])
    print("answer :", answer(result))


@tool
def part_stock(part: str) -> str:
    """Look up stock, storage location and name for a Veenhaven Bus part number."""
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute("SELECT part_no, name, stock, depot FROM parts WHERE part_no = ?",
                          (part.upper(),)).fetchall()
    return json.dumps(rows)


@tool
def fault_guidance(error_code: str) -> str:
    """Find the manual's meaning, first action and replacement condition for a bus error code."""
    return "\n".join(f"[{name}]\n{line}" for name, text in PAGES.items()
                     for line in text.splitlines()
                     if line.startswith("| " + error_code.upper() + " |")) or "Unknown error code."


STOCK_TASK = "Look up CE-CF-02 stock and location. Answer briefly from the tool."
MANUAL_TASK = "Look up F-CE-110. Give the first action and replacement condition, citing the file."
QUESTION = ("Bus CE12-2021-0117 shows F-CE-110. How many CE-CF-02 cooling fans are in stock, "
            "at which location, and what are the manual's first action and replacement condition? Be brief.")

In [3]:
from langchain_core.callbacks import BaseCallbackHandler

seen = []

class Sizes(BaseCallbackHandler):
    def on_chat_model_start(self, serialized, messages, **kwargs):
        seen.append(sum(len(batch) for batch in messages))

llm = make_llm(callbacks=[counter, Sizes()], max_tokens=1800, max_retries=0)
parts = {"name": "parts", "description": "Looks up stock and location for one part number.",
         "system_prompt": "Use part_stock. Return the stock and location.", "tools": [part_stock]}
team = create_deep_agent(model=llm, subagents=[parts], system_prompt=(
    "Call task with subagent_type parts. Put the part number in the description. Do not write files."))
task_tool = team.get_graph().nodes["tools"].data.tools_by_name["task"]
print("listed :", [line.strip() for line in task_tool.description.splitlines() if line.strip().startswith("- ")])
seen.clear()
result = measured("dict-subagent", lambda: team.invoke(ask(
    "How many CE-CF-02 are in stock, and where? Delegate to parts."), LIMIT))
show(result)
print("message counts :", seen)

listed : ['- general-purpose: General-purpose agent for researching complex questions, searching for files and content, and executing multi-step tasks. When you are searching for a keyword or file and are not confident that you will find the right match in the first few tries use this agent to perform the search for you. This agent has access to all tools as the main agent.', '- parts: Looks up stock and location for one part number.', '- Launch multiple agents concurrently when their tasks are independent, using a single message with multiple tool calls.', '- Each invocation is stateless by default: the agent sees only the prompt you give it and returns a single final report. Put full detail in the prompt and state exactly what it should return — unless an agent type below says it inherits your conversation instead.', "- The agent's report is not shown to the user; relay a summary yourself.", "- Tell the agent whether to create content, analyze, or only research, since it can't necess

metrics : {"label": "dict-subagent", "calls": 4, "input_tokens": 7544, "output_tokens": 107, "malformed": 0, "errors": 0, "seconds": 12.7}
tools : ['task']
answer : There are **6 CE-CF-02** in stock at **Rietpoort**.
message counts : [2, 2, 4, 4]


## b · compiled-subagent

CompiledSubAgent wraps a graph that already has a messages channel. The parent still calls it through task, and the graph does not inherit the harness middleware.

In [4]:
manuals_graph = create_agent(llm, tools=[fault_guidance],
    system_prompt="Use fault_guidance. Return the first action and the replacement condition.")
compiled = {"name": "manuals-graph", "description": "A compiled agent for one bus error code.",
            "runnable": manuals_graph}
parent = create_deep_agent(model=llm, subagents=[compiled], system_prompt=(
    "Call task with subagent_type manuals-graph and include the error code. Do not write files."))
seen.clear()
result = measured("compiled", lambda: parent.invoke(ask(
    "Bus CE12-2021-0117 shows F-CE-110. Delegate the first-action lookup."), LIMIT))
show(result)
print("message counts :", seen)

metrics : {"label": "compiled", "calls": 4, "input_tokens": 4516, "output_tokens": 191, "malformed": 0, "errors": 0, "seconds": 9.2}
tools : ['task']
answer : For F-CE-110, the first action is to **check the traction cooling fan**. Replace it if airflow is below 80%.
message counts : [2, 2, 4, 4]


## c · fork-mode

mode fork passes the parent messages into the subagent and appends the task description. The description can omit the error code because that code is already in the conversation. The user question is the one the compiled subagent received, so the two runs differ in the worker only.

In [5]:
forked = {"name": "manuals-fork", "description": "Continues this conversation and looks up the error already named.",
          "system_prompt": "Use fault_guidance on the error code in the conversation. Return the first action.",
          "tools": [fault_guidance], "mode": "fork"}
parent = create_deep_agent(model=llm, subagents=[forked], system_prompt=(
    "Call task with subagent_type manuals-fork. The description is: return the first action. "
    "Do not repeat the error code in the description. Do not write files."))
seen.clear()
result = measured("fork", lambda: parent.invoke(ask(
    "Bus CE12-2021-0117 shows F-CE-110. Delegate the first-action lookup."), LIMIT))
show(result)
print("message counts :", seen)
for message in result["messages"]:
    if getattr(message, "tool_calls", None):
        print("calls :", message.tool_calls)

metrics : {"label": "fork", "calls": 4, "input_tokens": 8761, "output_tokens": 97, "malformed": 0, "errors": 0, "seconds": 11.6}
tools : ['task']
answer : Check the traction cooling fan.
message counts : [2, 3, 5, 4]
calls : [{'name': 'task', 'args': {'description': 'Find and return the first action for bus CE12-2021-0117 showing F-CE-110. Do not write files.', 'subagent_type': 'manuals-fork'}, 'id': 'call_66QJvoQo8oammc3RGuvUsaCF', 'type': 'tool_call'}]


## Things to try

Optional : we can change the bus task and compare the answer, calls and tokens.